
# Vrijhof electricity consumption — data preparation

**Course:** Introduction to Data Science  
**Case:** Vrijhof building, University of Twente  
**Analysis period:** 1 September 2025 – 31 August 2026  
**Target:** `electricity_kWh`

This supporting notebook shows how the raw electricity and KNMI weather files are loaded, inspected, cleaned and combined.

The notebook is intentionally simple: it uses separate Pandas steps instead of custom functions, classes or preprocessing pipelines. The goal is that every important step can be explained during the poster session.

## Use of an LLM

An LLM was used as a **coding assistant** during development. It helped translate intended data-preparation steps into Pandas syntax and helped discuss alternative ways to handle data-quality issues.

For the important decisions below, the notebook distinguishes between:

- **What we found:** what we observed in the data;
- **What we asked the LLM:** where we asked for coding or methodological options;
- **Decision:** the choice used in the project;
- **Check:** how the result was verified.

These boxes summarize the development process; they are **not verbatim prompt transcripts**.



## 1. Imports and source files

Only Pandas and `Path` are needed for this data-preparation notebook. The academic-calendar PDF is listed for traceability, but its date ranges are entered manually below rather than automatically extracted from the PDF.


In [1]:
from pathlib import Path
import pandas as pd

folder = Path(".")

electricity_file = folder / "vrijhof_electricity.csv"
weather_file = folder / "knmi_weather.csv.txt"
academic_calendar_file = folder / "ACADEMIC CALENDAR 2025_2026 juli25.pdf"

start_time = pd.Timestamp("2025-09-01 00:00:00")
end_time = pd.Timestamp("2026-08-31 23:00:00")

print("Current folder:", Path.cwd())

for file in [electricity_file, weather_file, academic_calendar_file]:
    print(file, "->", file.exists())

Current folder: c:\Users\noahc\Downloads\VU_Q1\Project_ITD\Data
vrijhof_electricity.csv -> True
knmi_weather.csv.txt -> True
ACADEMIC CALENDAR 2025_2026 juli25.pdf -> True



## 2. Load and inspect the raw electricity data

Before cleaning, we inspect the file as it is supplied.


In [2]:

electricity_raw = pd.read_csv(electricity_file)

print("First rows:")
print(electricity_raw.head())

print("\nColumns:")
print(electricity_raw.columns)

print("\nShape:")
print(electricity_raw.shape)

print("\nMissing values:")
print(electricity_raw.isna().sum())

print("\nFully duplicated rows:")
print(electricity_raw.duplicated().sum())


First rows:
  x0 (Electricity (Vrijhof))  y0 (kWh)
0        2025-09-01 00:00:00        29
1        2025-09-01 00:00:00         0
2        2025-09-01 01:00:00        29
3        2025-09-01 02:00:00        28
4        2025-09-01 03:00:00        27

Columns:
Index(['x0 (Electricity (Vrijhof))', 'y0 (kWh)'], dtype='str')

Shape:
(8760, 2)

Missing values:
x0 (Electricity (Vrijhof))    0
y0 (kWh)                      0
dtype: int64

Fully duplicated rows:
0


In [3]:

electricity = electricity_raw.copy()

electricity = electricity.rename(columns={
    "x0 (Electricity (Vrijhof))": "timestamp",
    "y0 (kWh)": "electricity_kWh"
})

electricity["timestamp"] = pd.to_datetime(electricity["timestamp"])
electricity = electricity.sort_values("timestamp").reset_index(drop=True)

print("Info:")
electricity.info()

print("\nNumerical summary:")
print(electricity.describe())


Info:
<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   timestamp        8760 non-null   datetime64[us]
 1   electricity_kWh  8760 non-null   int64         
dtypes: datetime64[us](1), int64(1)
memory usage: 137.0 KB

Numerical summary:
                        timestamp  electricity_kWh
count                        8760      8760.000000
mean   2026-03-02 10:55:37.808219        68.066895
min           2025-09-01 00:00:00         0.000000
25%           2025-12-01 04:45:00        40.000000
50%           2026-03-02 10:30:00        63.000000
75%           2026-06-01 17:15:00        92.000000
max           2026-08-31 23:00:00       250.000000
std                           NaN        35.214621



## 3. Check electricity timestamps and unusual values

### Development decision

**What we found**  
The timestamp `2025-09-01 00:00:00` occurs twice, with two different values: 29 kWh and 0 kWh. The hourly sequence also misses the local 02:00 hour on 29 March 2026 because of the daylight-saving-time transition. Electricity values range up to relatively high values, but an unusual value is not automatically an error.

**What we asked the LLM**  
We asked how to make duplicate and missing timestamps visible without immediately deleting or replacing observations.

**Decision**  
We inspect the problems first. We do not automatically remove high or low electricity observations.

**Check**  
The cells below explicitly print duplicate timestamps, missing expected timestamps and the lowest/highest electricity values.


In [4]:

duplicate_electricity = electricity[
    electricity["timestamp"].duplicated(keep=False)
]

print("Duplicate electricity timestamps:")
print(duplicate_electricity)

expected_hours = pd.date_range(
    start=start_time,
    end=end_time,
    freq="h"
)

missing_electricity_hours = expected_hours[
    ~expected_hours.isin(electricity["timestamp"])
]

print("\nMissing electricity timestamps:")
print(missing_electricity_hours)


Duplicate electricity timestamps:
   timestamp  electricity_kWh
0 2025-09-01               29
1 2025-09-01                0

Missing electricity timestamps:
DatetimeIndex(['2026-03-29 02:00:00'], dtype='datetime64[us]', freq='h')


In [5]:

print("10 lowest electricity values:")
print(
    electricity[["timestamp", "electricity_kWh"]]
    .sort_values("electricity_kWh")
    .head(10)
)

print("\n10 highest electricity values:")
print(
    electricity[["timestamp", "electricity_kWh"]]
    .sort_values("electricity_kWh", ascending=False)
    .head(10)
)


10 lowest electricity values:
               timestamp  electricity_kWh
1    2025-09-01 00:00:00                0
2141 2025-11-29 04:00:00               15
6196 2026-05-17 04:00:00               15
6241 2026-05-19 01:00:00               16
2690 2025-12-22 01:00:00               16
2501 2025-12-14 04:00:00               16
4537 2026-03-09 00:00:00               16
2166 2025-11-30 05:00:00               16
3917 2026-02-11 04:00:00               16
4274 2026-02-26 01:00:00               16

10 highest electricity values:
               timestamp  electricity_kWh
7173 2026-06-26 21:00:00              250
7194 2026-06-27 18:00:00              250
7164 2026-06-26 12:00:00              249
7118 2026-06-24 14:00:00              247
7141 2026-06-25 13:00:00              246
7190 2026-06-27 14:00:00              244
7189 2026-06-27 13:00:00              239
7116 2026-06-24 12:00:00              237
7140 2026-06-25 12:00:00              233
7139 2026-06-25 11:00:00              232



## 4. Load and inspect the KNMI weather data

The KNMI file contains documentation lines beginning with `#`. The data itself has seven columns.

According to the header in the supplied KNMI file:

- station 290 is **Twenthe Airport**;
- `HH` is reported in **UT**;
- temperature is in **0.1 °C**;
- solar radiation is in **J/cm² per hourly interval**;
- precipitation is in **0.1 mm**, with `-1` meaning less than 0.05 mm;
- relative humidity is in **percent**.


In [6]:

weather_raw = pd.read_csv(
    weather_file,
    comment="#",
    header=None,
    skipinitialspace=True
)

print("First rows:")
print(weather_raw.head())

print("\nShape:")
print(weather_raw.shape)

print("\nColumns before renaming:")
print(weather_raw.columns)

print("\nMissing values:")
print(weather_raw.isna().sum())


First rows:
     0         1  2    3  4  5   6
0  290  20250831  1  165  0  0  70
1  290  20250831  2  169  0  0  70
2  290  20250831  3  170  0  0  70
3  290  20250831  4  172  0  0  70
4  290  20250831  5  173  0  0  71

Shape:
(8808, 7)

Columns before renaming:
Index([0, 1, 2, 3, 4, 5, 6], dtype='int64')

Missing values:
0    0
1    0
2    0
3    0
4    0
5    0
6    0
dtype: int64


In [7]:

weather = weather_raw.copy()

weather.columns = [
    "station",
    "date",
    "hour",
    "temperature_raw",
    "solar_radiation",
    "precipitation_raw",
    "relative_humidity"
]

print("Stations in the file:")
print(weather["station"].value_counts())

weather = weather[
    weather["station"] == 290
].copy()


Stations in the file:
station
290    8808
Name: count, dtype: int64



## 5. Convert KNMI units

### Development decision

**What we found**  
KNMI uses scaled values for temperature and precipitation. Precipitation value `-1` means a trace amount below 0.05 mm.

**What we asked the LLM**  
We asked how to convert the units while keeping the treatment of `-1` visible.

**Decision**  
Temperature is divided by 10. Trace precipitation (`-1`) is treated as 0 mm, and the remaining precipitation values are divided by 10.

**Check**  
We first count the `-1` values and then verify that the converted precipitation variable has no negative values.


In [8]:

weather["date"] = pd.to_datetime(
    weather["date"].astype(str),
    format="%Y%m%d"
)

weather["outdoor_temperature"] = (
    weather["temperature_raw"] / 10
)

print("Number of precipitation values equal to -1:")
print((weather["precipitation_raw"] == -1).sum())

print("\nExamples:")
print(
    weather.loc[
        weather["precipitation_raw"] == -1,
        ["date", "hour", "precipitation_raw"]
    ].head(10)
)


Number of precipitation values equal to -1:
808

Examples:
         date  hour  precipitation_raw
8  2025-08-31     9                 -1
9  2025-08-31    10                 -1
18 2025-08-31    19                 -1
19 2025-08-31    20                 -1
20 2025-08-31    21                 -1
35 2025-09-01    12                 -1
68 2025-09-02    21                 -1
69 2025-09-02    22                 -1
74 2025-09-03     3                 -1
76 2025-09-03     5                 -1


In [9]:

weather["precipitation_mm"] = weather["precipitation_raw"].copy()

weather.loc[
    weather["precipitation_mm"] == -1,
    "precipitation_mm"
] = 0

weather["precipitation_mm"] = (
    weather["precipitation_mm"] / 10
)

print("Weather summary after unit conversion:")
print(
    weather[
        [
            "outdoor_temperature",
            "solar_radiation",
            "precipitation_mm",
            "relative_humidity"
        ]
    ].describe()
)

print("\nNegative precipitation values:")
print((weather["precipitation_mm"] < 0).sum())


Weather summary after unit conversion:
       outdoor_temperature  solar_radiation  precipitation_mm  \
count          8808.000000      8808.000000       8808.000000   
mean             11.173808        46.458447          0.081312   
std               7.591225        74.332239          0.501029   
min             -10.300000         0.000000          0.000000   
25%               5.875000         0.000000          0.000000   
50%              11.050000         2.000000          0.000000   
75%              16.200000        66.000000          0.000000   
max              37.100000       318.000000         22.200000   

       relative_humidity  
count        8808.000000  
mean           78.959923  
std            17.839471  
min            17.000000  
25%            68.000000  
50%            84.000000  
75%            93.000000  
max           100.000000  

Negative precipitation values:
0



## 6. Convert KNMI time to Dutch local time

### Development decision

**What we found**  
The electricity timestamps are local clock times, while KNMI reports hourly intervals in UT. After conversion to Dutch local time, 26 October 2025 contains two observations at local 02:00 because the clock moves back. On 29 March 2026, local 02:00 does not exist because the clock moves forward.

**What we asked the LLM**  
We asked how to convert the KNMI hourly codes to proper UTC timestamps and then to `Europe/Amsterdam`, while keeping the daylight-saving-time issue visible.

**Decision**  
We construct the start of each KNMI hourly interval using `hour - 1`, localize it as UTC and convert it to Dutch local time.

**Check**  
The duplicate winter-time hour and missing summer-time hour are printed explicitly.


In [10]:

weather["timestamp_utc"] = (
    weather["date"]
    + pd.to_timedelta(weather["hour"] - 1, unit="h")
)

weather["timestamp_utc"] = (
    weather["timestamp_utc"]
    .dt.tz_localize("UTC")
)

weather["timestamp"] = (
    weather["timestamp_utc"]
    .dt.tz_convert("Europe/Amsterdam")
    .dt.tz_localize(None)
)

weather = weather[
    (weather["timestamp"] >= start_time) &
    (weather["timestamp"] <= end_time)
].copy()

weather = weather.sort_values("timestamp").reset_index(drop=True)


In [11]:

duplicate_weather = weather[
    weather["timestamp"].duplicated(keep=False)
]

print("Duplicate local KNMI timestamps:")
print(
    duplicate_weather[
        ["timestamp_utc", "timestamp"]
    ]
)

missing_weather_hours = expected_hours[
    ~expected_hours.isin(weather["timestamp"])
]

print("\nMissing local KNMI timestamps:")
print(missing_weather_hours)


Duplicate local KNMI timestamps:
                 timestamp_utc           timestamp
1322 2025-10-26 00:00:00+00:00 2025-10-26 02:00:00
1323 2025-10-26 01:00:00+00:00 2025-10-26 02:00:00

Missing local KNMI timestamps:
DatetimeIndex(['2026-03-29 02:00:00'], dtype='datetime64[us]', freq='h')



## 7. Handle the two ambiguous local timestamps

### Development decision

**What we found**

1. Electricity contains two conflicting values at `2025-09-01 00:00`.
2. KNMI contains two different weather observations that both become local `2025-10-26 02:00`.
3. The electricity data has no UTC offset that would tell us which winter-time weather observation belongs to its single 02:00 row.

**What we asked the LLM**  
We discussed options such as keeping one value, averaging, or excluding the ambiguous timestamps.

**Decision**  
We do not choose a value without source evidence. We exclude the complete local timestamp `2025-09-01 00:00` from electricity and exclude `2025-10-26 02:00` from both datasets. The original raw files remain unchanged.

**Check**  
After this step, both datasets must have unique timestamps.


In [12]:

print("Electricity rows at 2025-09-01 00:00:")
print(
    electricity.loc[
        electricity["timestamp"] ==
        pd.Timestamp("2025-09-01 00:00:00")
    ]
)

print("\nWeather rows at 2025-10-26 02:00:")
print(
    weather.loc[
        weather["timestamp"] ==
        pd.Timestamp("2025-10-26 02:00:00"),
        [
            "timestamp_utc",
            "timestamp",
            "outdoor_temperature",
            "solar_radiation",
            "precipitation_mm",
            "relative_humidity"
        ]
    ]
)


Electricity rows at 2025-09-01 00:00:
   timestamp  electricity_kWh
0 2025-09-01               29
1 2025-09-01                0

Weather rows at 2025-10-26 02:00:
                 timestamp_utc           timestamp  outdoor_temperature  \
1322 2025-10-26 00:00:00+00:00 2025-10-26 02:00:00                  6.1   
1323 2025-10-26 01:00:00+00:00 2025-10-26 02:00:00                  6.0   

      solar_radiation  precipitation_mm  relative_humidity  
1322                0               0.0                 86  
1323                0               0.0                 85  


In [13]:

ambiguous_timestamps = pd.to_datetime([
    "2025-09-01 00:00:00",
    "2025-10-26 02:00:00"
])

electricity_clean = electricity[
    ~electricity["timestamp"].isin(ambiguous_timestamps)
].copy()

weather_clean = weather[
    ~weather["timestamp"].isin(ambiguous_timestamps)
].copy()

print("Electricity shape:", electricity_clean.shape)
print("Weather shape:", weather_clean.shape)

print("\nDuplicate electricity timestamps:")
print(electricity_clean["timestamp"].duplicated().sum())

print("Duplicate weather timestamps:")
print(weather_clean["timestamp"].duplicated().sum())


Electricity shape: (8757, 2)
Weather shape: (8757, 11)

Duplicate electricity timestamps:
0
Duplicate weather timestamps:
0



## 8. Merge electricity and weather

### Development decision

**What we found**  
After removing the ambiguous timestamps, electricity and weather both have unique timestamp keys.

**What we asked the LLM**  
We asked which simple merge keeps electricity as the main dataset.

**Decision**  
We use a **left merge** from electricity to weather on `timestamp`.

**Check**  
The number of rows should stay unchanged, weather variables should have no missing values, and timestamps should remain unique.


In [14]:

weather_for_merge = weather_clean[
    [
        "timestamp",
        "outdoor_temperature",
        "solar_radiation",
        "precipitation_mm",
        "relative_humidity"
    ]
].copy()

merged = pd.merge(
    electricity_clean,
    weather_for_merge,
    on="timestamp",
    how="left"
)

print("Shape after merge:")
print(merged.shape)

print("\nMissing values:")
print(merged.isna().sum())

print("\nDuplicate timestamps:")
print(merged["timestamp"].duplicated().sum())


Shape after merge:
(8757, 6)

Missing values:
timestamp              0
electricity_kWh        0
outdoor_temperature    0
solar_radiation        0
precipitation_mm       0
relative_humidity      0
dtype: int64

Duplicate timestamps:
0



## 9. Add basic time variables

`hour_of_day` and `day_of_week` are derived directly from the timestamp using simple Pandas datetime operations.


In [15]:

merged["hour_of_day"] = merged["timestamp"].dt.hour
merged["day_of_week"] = merged["timestamp"].dt.day_name()

print(
    merged[
        ["timestamp", "hour_of_day", "day_of_week"]
    ].head()
)


            timestamp  hour_of_day day_of_week
0 2025-09-01 01:00:00            1      Monday
1 2025-09-01 02:00:00            2      Monday
2 2025-09-01 03:00:00            3      Monday
3 2025-09-01 04:00:00            4      Monday
4 2025-09-01 05:00:00            5      Monday



## 10. Add academic period and UT closure days

### Development decision

**What we found**  
The supplied University of Twente Academic Calendar distinguishes the four quartiles and shows the holiday period around Christmas/New Year, public holidays and the summer period.

**What we asked the LLM**  
We asked for a simple way to translate the calendar ranges into an hourly categorical variable and a 0/1 closure indicator.

**Decision**  
We enter the calendar ranges explicitly using `.loc`. For the project, the Christmas/New Year holiday period is separated as `winter_break`. The summer period is defined through 30 August 2026, so 31 August remains `outside_calendar`.

**Check**  
We print the category counts and the number of closed/open observations.


In [16]:

calendar_date = merged["timestamp"].dt.normalize()

merged["academic_period"] = "outside_calendar"

merged.loc[
    (calendar_date >= "2025-09-01") &
    (calendar_date <= "2025-11-09"),
    "academic_period"
] = "Q1"

merged.loc[
    (calendar_date >= "2025-11-10") &
    (calendar_date <= "2025-12-21"),
    "academic_period"
] = "Q2"

merged.loc[
    (calendar_date >= "2025-12-22") &
    (calendar_date <= "2026-01-04"),
    "academic_period"
] = "winter_break"

merged.loc[
    (calendar_date >= "2026-01-05") &
    (calendar_date <= "2026-02-01"),
    "academic_period"
] = "Q2"

merged.loc[
    (calendar_date >= "2026-02-02") &
    (calendar_date <= "2026-04-19"),
    "academic_period"
] = "Q3"

merged.loc[
    (calendar_date >= "2026-04-20") &
    (calendar_date <= "2026-07-05"),
    "academic_period"
] = "Q4"

merged.loc[
    (calendar_date >= "2026-07-06") &
    (calendar_date <= "2026-08-30"),
    "academic_period"
] = "summer_period"

print(merged["academic_period"].value_counts())


academic_period
Q4                  1848
Q3                  1847
Q2                  1680
Q1                  1678
summer_period       1344
winter_break         336
outside_calendar      24
Name: count, dtype: int64


In [17]:

ut_closed_dates = pd.to_datetime([
    "2025-12-25",
    "2025-12-26",
    "2025-12-29",
    "2025-12-30",
    "2025-12-31",
    "2026-01-01",
    "2026-01-02",
    "2026-04-03",
    "2026-04-06",
    "2026-04-27",
    "2026-05-05",
    "2026-05-14",
    "2026-05-15",
    "2026-05-25"
])

merged["UT_closed"] = (
    calendar_date.isin(ut_closed_dates)
    .astype(int)
)

print(merged["UT_closed"].value_counts())


UT_closed
0    8421
1     336
Name: count, dtype: int64



## 11. Add assessment period

### Development decision

**What we found**  
The official UT Academic Calendar distinguishes regular education from MSc exam periods and separate resit periods. These periods may provide additional context beyond the quartile variable.

**What we asked the LLM**  
We asked whether exam/resit periods could be represented as an additional simple variable without making the calendar processing unnecessarily complex.

**Decision**  
We add a binary variable called `assessment_period`:

- `0` = regular academic period / no exam or resit period indicated;
- `1` = period in which exams or resits are explicitly indicated in the UT calendar.

We use the broader term *assessment period* because the calendar also includes resits and because BSc and MSc schedules are not identical. This variable represents academic-calendar context; it does **not** mean that an exam actually took place in Vrijhof during every marked hour.

For the main academic year, the calendar shows MSc exam periods at the end of Q1, Q2, Q3 and Q4. The summer calendar also explicitly identifies week 30 as resits for fourth-quartile courses.

**Check**  
We print the number of observations in each category and inspect the first dates marked as an assessment period.


In [18]:

merged["assessment_period"] = 0

assessment_dates = (
    ((calendar_date >= "2025-10-27") &
     (calendar_date <= "2025-11-07"))
    |
    ((calendar_date >= "2026-01-19") &
     (calendar_date <= "2026-01-30"))
    |
    ((calendar_date >= "2026-04-13") &
     (calendar_date <= "2026-04-17"))
    |
    ((calendar_date >= "2026-06-22") &
     (calendar_date <= "2026-07-03"))
    |
    ((calendar_date >= "2026-07-20") &
     (calendar_date <= "2026-07-24"))
)

merged.loc[
    assessment_dates,
    "assessment_period"
] = 1

print("Assessment-period counts:")
print(merged["assessment_period"].value_counts())

print("\nFirst observations marked as assessment period:")
print(
    merged.loc[
        merged["assessment_period"] == 1,
        ["timestamp", "academic_period", "assessment_period"]
    ].head(20)
)


Assessment-period counts:
assessment_period
0    7653
1    1104
Name: count, dtype: int64

First observations marked as assessment period:
               timestamp academic_period  assessment_period
1342 2025-10-27 00:00:00              Q1                  1
1343 2025-10-27 01:00:00              Q1                  1
1344 2025-10-27 02:00:00              Q1                  1
1345 2025-10-27 03:00:00              Q1                  1
1346 2025-10-27 04:00:00              Q1                  1
1347 2025-10-27 05:00:00              Q1                  1
1348 2025-10-27 06:00:00              Q1                  1
1349 2025-10-27 07:00:00              Q1                  1
1350 2025-10-27 08:00:00              Q1                  1
1351 2025-10-27 09:00:00              Q1                  1
1352 2025-10-27 10:00:00              Q1                  1
1353 2025-10-27 11:00:00              Q1                  1
1354 2025-10-27 12:00:00              Q1                  1
1355 2025-10-27 13:00


## 12. Add scheduled building opening

### Development decision

**What we found / source used**  
The project uses the regular Vrijhof opening schedule:
- Monday–Friday: 08:00–24:00;
- Saturday–Sunday: 10:00–20:00;
- `UT_closed` dates override the regular schedule.

**What we asked the LLM**  
We asked how to convert these rules into a simple binary variable.

**Decision**  
`building_open` represents **scheduled opening status**, not measured occupancy.

**Check**  
We inspect the total number of open and closed hourly observations.

> The final report should cite the official Vrijhof opening-hours webpage, because that webpage is not included among the files used by this notebook.


In [19]:

merged["building_open"] = 0

weekday_open = (
    (merged["timestamp"].dt.dayofweek <= 4) &
    (merged["hour_of_day"] >= 8)
)

weekend_open = (
    (merged["timestamp"].dt.dayofweek >= 5) &
    (merged["hour_of_day"] >= 10) &
    (merged["hour_of_day"] < 20)
)

merged.loc[
    weekday_open | weekend_open,
    "building_open"
] = 1

merged.loc[
    merged["UT_closed"] == 1,
    "building_open"
] = 0

print(merged["building_open"].value_counts())


building_open
1    4992
0    3765
Name: count, dtype: int64



## 13. Create the previous-day electricity baseline

### Development decision

**What we found**  
A simple `shift(24)` means “24 rows earlier”. Because the dataset contains missing or deliberately excluded clock hours, 24 rows earlier is not always the same local hour on the previous calendar day.

**What we asked the LLM**  
We asked for a transparent method that matches by actual timestamp rather than row position.

**Decision**  
We copy `timestamp` and `electricity_kWh`, move the timestamps forward by one calendar day, rename the electricity column and merge it back.

**Check**  
The missing baseline timestamps are printed rather than silently filled.


In [20]:

previous_day = merged[
    ["timestamp", "electricity_kWh"]
].copy()

previous_day["timestamp"] = (
    previous_day["timestamp"]
    + pd.Timedelta(days=1)
)

previous_day = previous_day.rename(columns={
    "electricity_kWh":
    "electricity_previous_day_same_hour"
})

merged = pd.merge(
    merged,
    previous_day,
    on="timestamp",
    how="left"
)

missing_baseline = merged[
    merged["electricity_previous_day_same_hour"].isna()
]

print("Number of missing previous-day values:")
print(len(missing_baseline))

print("\nTimestamps with missing previous-day value:")
print(
    missing_baseline[
        ["timestamp", "electricity_kWh"]
    ]
)


Number of missing previous-day values:
26

Timestamps with missing previous-day value:
               timestamp  electricity_kWh
0    2025-09-01 01:00:00               29
1    2025-09-01 02:00:00               28
2    2025-09-01 03:00:00               27
3    2025-09-01 04:00:00               29
4    2025-09-01 05:00:00               30
5    2025-09-01 06:00:00               45
6    2025-09-01 07:00:00               57
7    2025-09-01 08:00:00               83
8    2025-09-01 09:00:00              107
9    2025-09-01 10:00:00              110
10   2025-09-01 11:00:00              128
11   2025-09-01 12:00:00              132
12   2025-09-01 13:00:00              122
13   2025-09-01 14:00:00              140
14   2025-09-01 15:00:00              145
15   2025-09-01 16:00:00              138
16   2025-09-01 17:00:00              129
17   2025-09-01 18:00:00              117
18   2025-09-01 19:00:00              113
19   2025-09-01 20:00:00              104
20   2025-09-01 21:00:00       


### Interpretation of the 26 baseline missings

The 26 missing values are expected:

- 23 retained hours on 1 September (`01:00`–`23:00`) have no previous day in the supplied analysis period;
- `2 September 00:00` has no baseline because `1 September 00:00` was excluded as ambiguous;
- `27 October 02:00` has no baseline because `26 October 02:00` was excluded as ambiguous;
- `30 March 02:00` has no baseline because local `29 March 02:00` does not exist during the switch to summer time.

These values are left missing rather than imputed.



## 14. Create the final dataset and perform final checks

The temporary preparation columns are left out. The final checks show the structure that is reported in Chapter 3.


In [21]:

final_dataset = merged[
    [
        "timestamp",
        "electricity_kWh",
        "outdoor_temperature",
        "solar_radiation",
        "precipitation_mm",
        "relative_humidity",
        "hour_of_day",
        "day_of_week",
        "academic_period",
        "assessment_period",
        "UT_closed",
        "building_open",
        "electricity_previous_day_same_hour"
    ]
].copy()

print("FINAL SHAPE:")
print(final_dataset.shape)

print("\nDATATYPES:")
print(final_dataset.dtypes)

print("\nMISSING VALUES:")
print(final_dataset.isna().sum())

print("\nDUPLICATE TIMESTAMPS:")
print(final_dataset["timestamp"].duplicated().sum())

print("\nFIRST ROWS:")
print(final_dataset.head())

print("\nLAST ROWS:")
print(final_dataset.tail())


FINAL SHAPE:
(8757, 13)

DATATYPES:
timestamp                             datetime64[us]
electricity_kWh                                int64
outdoor_temperature                          float64
solar_radiation                                int64
precipitation_mm                             float64
relative_humidity                              int64
hour_of_day                                    int32
day_of_week                                      str
academic_period                                  str
assessment_period                              int64
UT_closed                                      int64
building_open                                  int64
electricity_previous_day_same_hour           float64
dtype: object

MISSING VALUES:
timestamp                              0
electricity_kWh                        0
outdoor_temperature                    0
solar_radiation                        0
precipitation_mm                       0
relative_humidity                      


### Final result

The expected final structure is:

- **8,757 hourly observations**;
- **13 variables**, including the added `assessment_period` indicator;
- no duplicate timestamps;
- no missing electricity or weather values;
- **26 expected missing values** in `electricity_previous_day_same_hour`.

Unusual electricity values are retained because they were not shown to be measurement errors.


In [22]:

output_file = folder / "vrijhof_final_dataset_from_clean_notebook.csv"

final_dataset.to_csv(
    output_file,
    index=False
)

print("Saved as:")
print(output_file)


final_dataset.to_csv(
    "../Poster/final_dataset.csv",
    index=False
)

Saved as:
vrijhof_final_dataset_from_clean_notebook.csv


## Data sources

The following data sources were used in this project:

- **Electricity consumption:** [University of Twente Energy Data Platform](https://energydata.utwente.nl/)  
  Hourly electricity consumption data for the Vrijhof building.

- **Weather data:** [KNMI Hourly Weather Data](https://daggegevens.knmi.nl/klimatologie/uurgegevens)  
  Hourly weather observations from KNMI station 290 (Twenthe).

- **Academic calendar:** [University of Twente Academic Calendar](https://www.utwente.nl/en/service-portal/educational-support/planning-schedules/academic-calendar-year-cycle-and-holidays)  
  Used to derive academic periods, assessment periods, and university closing days.

- **Building opening hours:** [University of Twente – Opening Hours of Buildings](https://www.utwente.nl/en/service-portal/campus/opening-hours-of-buildings)  
  Used to derive the scheduled `building_open` variable.